# HGND — Sensitivity study

Interactive driver for the cross-dataset neutron-yield comparison.
For each dataset (`zeroSpot` / `defaultSpot` / `bigSpot`) corresponding to nuclear symmetry potentials 0 / 18 / 90 MeV, we compute

$$N_{n}(E_{k}; d) \;=\; \frac{N_{n,\text{reco}}(E_{k}; d)}{\varepsilon_{n}(E_{k}; d)}$$

and compare across `d`. If the three curves separate, the pipeline is sensitive to the symmetry potential. Runs on a single dataset today (defaultSpot) — extend to all three when the other datasets are transferred.

In [ ]:
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

PACKAGE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PACKAGE_ROOT not in sys.path:
    sys.path.insert(0, PACKAGE_ROOT)

from HGNDRecoGNN.analysis import metrics, efficiency, sensitivity
from HGNDRecoGNN.analysis.sensitivity import DatasetRun, compare_datasets, default_ekin_bins

## 1. Load per-dataset predictions

In [ ]:
# Point each entry at the pkl produced by scripts/evaluate.py.
# Comment out datasets that are not yet available.
PRED_PATHS = {
    'zeroSpot':    None,
    'defaultSpot': os.path.join(os.getcwd(), 'results', 'pred_clusters_smash.pkl'),
    'bigSpot':     None,
}

runs = []
for name, path in PRED_PATHS.items():
    if not path or not os.path.exists(path):
        print(f'skip {name}: {path}')
        continue
    df = pd.read_pickle(path)
    print(f'{name:15s} {len(df):>9,} clusters   {path}')
    runs.append(DatasetRun(name=name, clusters_df=df))

assert runs, 'no prediction files found — run scripts/evaluate.py first'

## 2. Per-dataset efficiency

In [ ]:
bins = default_ekin_bins()
print(f'{len(bins)-1} Ekin bins: [{bins[0]:.3f}, {bins[-1]:.2f}] GeV, log-spaced')

fig, ax = plt.subplots(figsize=(8, 4))
for run in runs:
    df = efficiency.epsilon_neutron_vs_ekin(run.clusters_df, bins, threshold=0.5)
    lbl = f'{run.name}' + (f' ({run.potential_mev:.0f} MeV)' if run.potential_mev is not None else '')
    ax.errorbar(df['ekin_mid'], df['epsilon'],
                yerr=[df['epsilon'] - df['epsilon_lo'], df['epsilon_hi'] - df['epsilon']],
                marker='o', label=lbl)
ax.set_xscale('log'); ax.set_xlabel('True Ekin [GeV]'); ax.set_ylabel(r'$\varepsilon_{n}(E_k)$')
ax.set_title('Reconstruction efficiency vs Ekin  (threshold 0.5)')
ax.grid(True, which='both', alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 3. Solved neutron yield across datasets

In [ ]:
tables = compare_datasets(runs, bins, threshold=0.5)
for name, df in tables.items():
    if name == 'combined':
        continue
    print(f'\n=== {name} ===')
    print(df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for run in runs:
    df = tables[run.name]
    lbl = f'{run.name}' + (f' ({run.potential_mev:.0f} MeV)' if run.potential_mev is not None else '')
    ax.errorbar(df['ekin_mid'], df['n_true_solved'],
                yerr=df['n_true_solved_err'], marker='o', capsize=2, label=lbl)
ax.set_xscale('log'); ax.set_xlabel('Ekin [GeV]'); ax.set_ylabel(r'$N_n^{\rm solved}(E_k)$')
ax.set_title('Neutron yield vs Ekin — sensitivity across symmetry potentials')
ax.grid(True, which='both', alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 4. Auxiliary metrics

In [ ]:
# Sanity: ROC / PR / purity+efficiency vs threshold for each dataset.
for run in runs:
    curves = metrics.classifier_curves(run.clusters_df['cl_score'].values,
                                       run.clusters_df['cl_label'].values)
    print(f'{run.name:15s} ROC-AUC {curves.roc_auc:.4f}  PR-AP {curves.pr_ap:.4f}')
    ep = metrics.efficiency_purity_vs_threshold(run.clusters_df)
    display(ep.round(3))